## LECTURE 4: Stack & Procedures -- The #1 Exam Topic
   This lecture is the linchpin of Part II. Every past paper has a question
   where you're given assembly and need to reconstruct the C function. To do 
   that, you need to understand four things deeply: the stack, `call`/`ret`,
   the calling convention, and register saving. ...


1. The Stack -- A Region of Memory
   The stack is just a chunk of memory that grows DOWNWARD (toward lower 
   addresses). `%rsp` always points to the "top" of the stack -- but since it
   grows downwards, "top" means the lowest address currently in use.

   Two fundamental operations:

   `pushq %reg` does two things atomically: (1) decrement `%rsp` by 8, then (2)
   write the register's value to the memory at the new `%rsp`. It's equivalent
   to `subq $8, %rsp` followed by `movq %reg, (%rsp)`.

   `popq %reg` is the reverse: (1) read the value at `(%rsp)` into the register,
   then (2) increment `%rsp` by 8. Equivalent to `movq (%rsp), %reg` followed by
   `addq $8, %rsp`.

   Why 8? because in x86-64, addresses are 4 bits = 8 bytes. The stack is 
   primarily used to store addresses (return addresses, saved registers), so
   everything is aligned to 8 bytes.


2. call and ret -- How Functions Transfer Control
   `call label` does two things: (1) push the address of the next instruction
   (the return address) onto the stack, then (2) jump to `label`. So if you have
   :

```asm
40050a:  callq 400502 <add>
40050f:  mov %eax, (%rbx)
```
   The `call` pushes `0x40050f` onto the stack (the address of the `mov`
   instrcution), then jumps to `0x400502`.

   `ret` is the reverse: pop the top of the stack into `%rip` (the program 
   counter). So it reads `0x40050f` from the stack, increments `%rsp` by 8, and 
   jumps to that address. Execution continues with the `mov` instrcution. 

   This is exactly like the JAL/JALR mechanism from RISC-V part I, except RISC-V
   stores the return address in a register (`ra`/`x1`) while x8 stores it on the
   stack. The stack approach is more powerful becuase it naturally supports 
   arbitrary nesting -- each `call` pushes another return address, and each `ret`
   pops the right one. 


3. The Calling Convention -- The Contract between Caller and Callee
   
   ...

   1 - %rdi
   2 - %rsi
   3 - %rdx
   4 - %rcx
   5 - %r8
   6 - %r9

   Any additional arguments (7th, 8th, ...) are pushed onto the stack in reverse
   order. The return value goes in `%rax`.

   MEMORISE THIS ORDER. When you see assembly and need to figure out what the
   function's parameters are, the very first thing you do is look at which of 
   these registers the function uses without writing to them first. If the
   first instruction is `addl %edi, %esi`, you immediately know this function
   takes at least two `int` arguments.

   The register naming also tells you the types: `%edi` (32-bit `e` prefix) 
   means `int`, `%rdi` (64-bit `r` prefix) means `long` or a pointer, ...
   `di` means short:16-bit... `%dil` means char:8-bit...


4. REGISTER SAVING -- Caller-Saved vs Callee-Saved
   Here's the problem: caller and callee share the same 16 registers. If `foo`
   has a value in `%rdx` that it needs after calling `bar`,    

   ...

2. Caller-Saved vs. Callee-Saved Registers
   
   Think of the CPU's 16 registers as a shared whiteboard. When Function A (the
   Caller) temporarily pauses to let Function B (the Callee) run, they have to
   agree on who cleans up the whiteboard.
      - Caller-Saved ("Trashable"): These are scratchpads. Function B is allowed
        to overwrite them immediately. If Function A needs the data in these 
        registers after Function B finishes, Function A better save it somewhere
        else first. 
         * Includes: `%rax`, `%rdi`, `%rsi`, `%rdx`, `%rcx`, `%r8`, `%r9`, `%r10`, `%r11`
      - Callee-Saved ("Sacred"): Function B is allowed to use these, but it MUST
        take a picture of what was there, use the space, and then erase and 
        redraw the original data before returning to Function A.
         * Includes: `%rbx`, `%rbp`, `%r12`-`%r15`


---
The Ultimate Exam Example
   ...: stashing a parameter into a callee-saved register to survive the 
   function call.


THE C CODE:
```c
long calculate_total(long x) {
   long y = helper_func(x);
   return x + y;              // We need `x` AFTER the function call!
}
```
   
   THE PROBLEM: When `calculate_total` starts, `x` is living in `%rdi`. To call
   `helper_func(x)`, we leave `x` in `%rdi`. But `helper_func` is allowed to 
   completely trash `%rdi` while it runs! How do we keep `x` safe so we can add
   it to `y` later?


THE ASSEMBLY SOLUTION:
```asm
calculate_total:
    pushq %rbx       # 1. We want to use %rbx to keep `x` safe.
                     #    Since %rbx is callee-saved, WE must save the old value first

    movq %rdi, %rbx  # 2. Stash `x` safely into %rbx
    call helper_func # 3. Call the helper. %rdi might get destroyed here. The 
                     #    result `y` comes back in %rax.

    addq %rbx, %rax  # 4. Add our safely stored `x` (%rbx) to `y` (%rax). %rax
                     #    now holds the final return value.

    popq %rbx        # 5. Restore the original %rbx we saved in step 1. 
    ret              # 6. Return to whoever called us. 
```

   WHY THIS MATTERS: When you are reading assembly in an exam and see 
   `pushq %rbx` followed immediately by `movq %rdi, %rbx`, you instantly know 
   the logic: "Ah, this function is going to call another function, and it needs
   to remember the first argument for later."

```asm
apply_discount:
    pushq %rbx
    movq  %rdi, %rbx
    call  calculate_discount

    # movq  %rbx, %rsi
    # subq  %rax, %rsi
    # movq  %rsi, %rax
    subq  %rbx, %rax
    negq  %rax

    popq  %rbx
    ret   
```

```asm
double_and_add:
    pushq %rsi
    call  double_and_add
    popq  %rdx
    addq  %rdx, %rax
    ret
```

```asm
compute_combo:  
    leaq  (%edi, %esi), %ecx
    pushq %ebx
    movq  %ecx, %ebx
    call  multiply_them 
    addq  %ebx, %eax
    popq  %ebx
    ret
```    

```asm
count_up:     
    movq  $0, %eax
    jmp   .L_test
.L_loop:    
    incl  %rax
.L_test:
    cmpq  %rdi, %rax
    jl    .L_loop
    ret
```

```asm
sum_down:       // %edi = n // %rax = sum
    movl  $0, %rax
.L_loop:
    addq  %rdi, %rax
    decl  %rdi
.L_test:
    cmpq  $0, $rdi
    jg    .L_loop
    ret
```

---

-- ... Those things with the colons at the end are called LABELS.

   Think of a label as a "bookmark" in your code. It doesn't actually do 
   anything on its own; it just gives a name to a specific memory address so 
   that your `jmp` (jump) instructions know where to go.

   Here is the breakdown of the conventions for naming these labels, 
   specifically the `.` and the `L`.


1. THE DOT (`.`): Local vs. Global
   In the GNU Assembler (the standard for Linux/AT&T syntax), the dot `.` at the 
   start of a label means it is a LOCAL SYMBOL.
      * GLOBAL LABELS (No Dot): Labels like `main:` or `calculate_total:` are 
        global. This means the Linker can see them, and other C files or
        assembly files can call them.
      * LOCAL LABELS (With Dot): Labels like `.loop:` or `.continue:` are local.
        They are "invisible" outside of the current file.


WHY IS THE DOT SO IMPORTANT?
   Imagine you write two different functions in the same file, and both of them
   have a while loop If you just named the label `loop:` for both of them, the
   assembler would crash and say... Error... By using local labels, you keep the
   "global namespace" clean so you don't accidentally create naming collisions.


2. THE `L` (`.L_loop`): The Compiler's Fingerprint
   The `L` stands for "Local" (or sometimes just "Label").

   When you write C code and ask GCC to compile it into assembly, the compiler
   has to automatically generate dozens of labels for all your `if` statements, 
   `for` ... 

   GCC's standard convention is to name these auto-generated labels starting 
   with `.L`:
      - ... will often see things like `.L2`, `.L3`, `.L4` in raw compiler 
        output.
      - When humans write assembly... they usually adapt this convention but 
        make it readable, like `.L_loop` or `.L_test`...


SUMMARY BY EXAMPLE
...

```asm
   .global my_function        # Tells the linker to make this visible globally

my_function:                  # GLOBAL LABEL: Other files can `call` this
    movl  $0, %eax
.L_loop:    
    incq  %rax
.L_test:
    cmpq  %rdi, %rax
    jl    .L_loop
    ret    
```

-- In x86-64 assembly language, `%rip` refers to the INSTRCUTION POINTER 
   REGISTER (or program counter), which holds the memory address of the next 
   instruction to be executed by the CPU. While it is a specialised 64-bit 
   register, its most common appearance in modern assembly code is in 
   RIP-relative addressing, which is used to access global variables and data. 

-- ... JAL (Jump and Link) is an instruction used to call functions and 
   subroutines. It performs two key actions: it jumps to a target address and
   saves the address of the next instruction (return address) into a register
   (ra), allowing the program to return later. 
        (RISC-V and MIPS)   

---

   ... this section ... is the absolute holy grail for reverse-engineering
   x86-64 assembly. If you understand how functions share the CPU and memory, 
   you can trace almost any program.

   ...


4. Register Saving: The "Shared Workspace" Problem  
   
   ...

EXAM PATTERN RECOGNITION
   If you look at the first few lines of an assembly function and see:

```asm
my_function:
    pushq %rbx
    pushq %r12
    movq  %rdi, %rbx
```
   You immediately know three things:
      1. This function is going to call another function soon.
      2. It needs to keep the 1st argument (`%rdi`) safe during that upcoming
         call.
      3. It moves `%rdi` into `%rbx` because `%rbx` is sacred (callee-saved)
         and won't be destroyed by the upcoming call.   


---
5. STACK FRAMES: The Function's Private Desk
   While registers are the shared whiteboard, the STACK FRAME is a function's
   private desk.

   in x86-64, the stack grows DONWARD in memory. The top of the stack is 
   traced by the Stack Point register: `%rsp`.


HOW A FRAME IS BORN AND DIES
   When a function needs a desk, it "pulls the ceiling down" by subtracting
   from `%rsp`.

```asm
complex_func:
    subq $32, $rsp          # Allocates 32 bytes of private memory
    # ... does work using memory addresses like 8(%rsp) or 16(rsp)
    addq $32, %rsp          # Destroys the desk by pushing the ceiling back up
    ret
```


WHAT GOES IN THE STACK FRAME?
   Registers are extremely fast, so the CPU prefers to se them for everything.
   A function only bothers building a slow, memory-based stack frame if it is
   forced to. It is forced to when:

   1. SAVING SACRED REGISTERS: Those `pushq %rbx` instructions... `pushq`...
      subtracts 8 from `%rsp` and writes the register's value into that memory
      space.
   2. SPILLING LOCAL VARIABLES: If your C code has 20 local `int` variables, the
      CPU doesn't have 20 free registers. The leftovers "spill" onto the stack.
   3. ARRAYS AND STRUCTS: Registers hold a single 64-bit value. You can't fit
      a 100-element array in a register, so it must live on the stack.
   4. The `&` Operator: This is a massive concept for your course. Registers do
      not have memory addresses. You cannot point to `%rax`. If yur C code does
      `int x = 5; int* ptr = &x;`, the compiler is forced to put `x` in the 
      stack frame so it actually has a memory address (like ...) to give to 
      `ptr`.


THE "LEAF" EXCEPTION
   If a function is a "leaf" (it calls absolutely nothing else), and it only
   uses a few simple variables, the compiler skips the stack frame entirely.

```c
int add(int a, int b) { return a + b; }
```   

```asm
add:
    leal (%rdi, %rsi), %eax
    ret
```



   ... essentially the "boss fight"... Translating C to assembly is one thing,
   but reverse-engineering Assembly back into C requires you to play detective
   with the CPU's state.

   ... 

   ... unpack the `call_incr` example to see why the compiler made the choices 
   it did, specifically focusing on the most common trap students fall into:
   THE ADDRESS-OF (`&`) RULE.


THE DETECTIVE WORK: Breaking Down `call_incr`
   Here is the setup: The function takes one argument (`x`), creates a local
   variable (`v1 = 21`), calls `increment(&v1, 50)`, and adds `x` to the result.

   Why does the assembly look so complicated for such a simple function?


1. WHY USE `%rbx` at all? (The Survival Problem)
   The function receives `x` in `%rdi`. But it immediately has to call 
   `increment`.

   * If `x` stays in `%rdi`, `increment` is allowed to destroy it...
   * ...


2. WHY ALLOCATE 16 BYTES ON THE STACK? (The Address-Of Trap)
   This is the most crucial insight in this snippet. Why didn't the compiler
   just put `v1 = 21` into a fast register like `%rcx`?

   Look at the C code: 




---

   ... Assembly can easily turn into an unreadable wall of text if it's not
   explained as a narrative.

   When you get a "translate this assembly back to C" question on your exam, 
   you are basically playing a detective at a crime scene The compiler left
   behind clues, and your job is to figure out what the original C code looked
   like

   ... followed by a narrative walkthrough of the `call_incr` example.


---
THE 6-STEP DETECTIVE FRAMEWORK

1. The Inputs (Who is arriving?)
   Before the function even does anything, look at which registers it reads from
   . The System V ABI strictly dictates the order: `%rdi`, `%rsi`, `%rdx`, 
   `%rcx`, `%r8`, `%r9`.

   - If the code uses `%rdi`, the C function has at least 1 argument.
   - If it uses `%edi` (the 32-bit version), that argument is an `int`.
   - If it uses `%edi` (the 64-bit version), it's a `long` or a pointer.


2. THE SETUP (The Prologue)
   Functions usually start by preparing their workspace.

   * `pushq %rbx`: The function is saying, "I need to use a callee-saved 
     (sacred) register, so I'm saving the previous owner's data to the stack."
   * `subq $N, %rsp`: The function is saying, "Registers aren't enough. I need
     $N$ bytes of actual RAM (a stack frame) to store local variables."


3. THE SURVIVAL STASH
   If you see an input argument immediately moved into a callee-saved register
   (e.g., `movq %rdi, %rbx`), the compiler is screaming a massive clue at you:
   "I am about to call another function, and I need to keep thios argument safe
   from being destroyed!"


4. THE FUNCTION CALL PREP
   Right before you see a `call` instruction, look at the 2 or 3 lines above it.
   The code will be rapidly stuffing values into `%rdi`, `%rsi`, etc. Those are
   arguments being passed to the new function. The moment the `call` finishes, 
   the new function's answer will be sitting in `%rax`.


5. THE FINAL OUTPUT
   Whatever value is sitting in `%rax` at the very bottom of the function, right
   before it exits, is what goes in your `return ___;` statement in C.


6. THE CLEANUP (The Epilogue)
   The function must put the CPU back exactly how it found it. You will see
   `addq $N, %rsp` (destroying the stack frame) and `popq %rbx` (restoring the
   previous owner's data). You can mostly ignore this part for C translations;
   it just confirms the function is ending correctly.


---
TRANSLATING `call_incr`: A Walkthrough
   Let's look at the assembly block and read it like a story using our
   framework.

```asm
call_incr:
    # --- PROLOGUE (Setup) ---
    pushq %rbx
    subq  $16, %rsp                 # <-- 16 bytes have been allocated for stack pointer

    # --- THE SURVIVAL STASH ---
    movq  %rdi, %rbx

    # --- LOCAL VARIABLES ---
    movl  $21, 8(%rsp)              # <-- long (8 bytes) moved into stack

    # --- FUNCTION CALL ---
    movl  $50, %esi
    leaq  8(%rsp), %rdi
    call  increment

    # --- THE MATH & RETURN ---
    addq  %rbx, %rax

    # --- EPILOGUE (Cleanup) ---
    addq  $16, %rsp
    popq  %rbx
    ret
```

long call_incr(long x) {
    long v1 = 21;
    long v2 = increment(&v1, 50);
    return x + v2;
}

```asm
mystery_one:
    movq    %rsi, %rax       
    cmpq    %rsi, %rdi       
    jl      .L_done          
    movq    %rdi, %rax       
    subq    %rsi, %rax       
.L_done:
    ret
```


```c
long mystery_one(long x, long y) {
    return x < y ? y : (x - y)
}
```

```asm
mystery_two:
    pushq   %rbx             
    subq    $16, %rsp        
    movq    %rdi, %rbx       
    movq    $50, 8(%rsp)     
    movq    %rbx, %rsi       
    leaq    8(%rsp), %rdi    
    call    process          
    addq    %rbx, %rax       
    addq    $16, %rsp        
    popq    %rbx             
    ret
```

long mystery_two(long x) {
    long v1 = 50;
    long v2 = process(&v1, x)
    return x + v2;
}

long process(long* x, long y);

```asm
mystery_three:
    movl    $0, %eax         
    movl    $0, %ecx         
    jmp     .L_test
.L_loop:
    addq    (%rdi,%rcx,8), %rax  
    incq    %rcx                 
.L_test:
    cmpq    %rsi, %rcx       
    jl      .L_loop
    ret
```

```c
long mystery_three(long *arr, long n) {
    long sum = 0, i = 0;
    while (i < n) {
        sum += arr[i]
        i++;
    }
    return temp;
}
```

---

---

---

## Lecture 5: Arrays & Structs in Assembly
   Two topics here: how the compiler turns array indexing into address 
   arithmetic, and how structs are laid out in memory with alignment padding. 
   Struct alignment is the more exam-tested of the two.


ARRAYS -- IT'S ALL ADDRESS MATH
   A 1D array `T A[L]` is just a contiguous block of `L * sizeof(T)`. Accessing
   `A[i]` means computing `base + i * sizeof(T)` and loading from that address.
   This maps directly to x86 scaled indexed addresing:

```c
int A[5];           // each element 4 bytes
// A[i] lives at address A + 4*i
```

```asm
# %rdi = A (base pointer), %rsi = i (index)
movl (%rdi, %rsi, 8), %eax          # eax = Mem[A + 4*i] = A[i]
```
   
   The scale factor (4 here) is always the element size. That's why the
   addressing mdoe only allows scales of 1, 2, 4, 8 -- they match `char`, 
   `short`, `int`, `long`.


---
2D NESTED ARRAYS (Row-Major)
   `int A[R][C]` is a single contiguous block in row-major order -- al of row 0
   first, then row 1, etc. Accessing `A[i][j]` means:

   Address = A + (i * C + j) * sizeof(int) = A + (i * C + j) * 4

   The compiler breaks this into lea tricks. For example with a 5-column array 
   of ints:

```asm
# i in %rdi, j in %rsi
leaq (%rdi, %rdi, 4) %rax       # rax = 5 * i
addq %rax, %rsi                 # rsi = 5 * i + j
movl A(,%rsi,4), %eax           # eax = Mem[A + 4 * (5*i + j)]
```
   The key exam insight: ONE MEMORY ACCESS IS ENOUGH becuase the entire 2D
   array is contiguous.

### 1. Multi-Level Arrays (The "Array of Pointers")
   There is a massive difference between a 2D array (`int matrix[3][3]`) and an
   array of pointers (`int *univ[3]`).

The Contiguous 2D Array (One Big Block):
   Imagine a bookcase. Every book is right next to the other. If the CPU wants
   row 2, column 1, it just does a quick math formula: 
      `Base Address + (Row * Width + Column) * Element Size`

   - ASSEMBLY CLUE: You will see a `lea` instruction or a multiplication doing
     that exact math, followed by a SINGLE memory access to grab the value. 


The Mult-Level Array (The Scanvenger Hunt):
   Imagine a drawer full of library cards. `univ[3]` is just an array of 3 cards
   (pointers). To find `univ[1][2]`, the CPU has to go on a scavenger hunt:

   1. ACCESS 1: Go to the `univ` array, grab pointer #1. (The CPU now has a 
      memory address).
   2. ACCESS 2: Travel to that new memory address, jump forward by 2 elements,
      and grab the data.
   -  ASSEMBLY CLUE: You will see TWO DISTINCT MEMORY READS (usually two `mov`
      instructions with parentheses). The first grabs the 8-byte pointer
      (`univ(, %rdi, 8)`), and the seocnd uses that pointer to fetch the final
      `int`.


---
2. STRUCTS -- Layout, Alignment, and Padding
   This is the classic exam trap Compilers are obsessed with efficiency. A CPU
   reads memory in "chunks" (like 4 bytes, or 8 bytes at a time). If a 4-byte
   `int` is sitting across the boundary of two different chunks, the CPU has to
   do double the work to read it. 

   To prevent this, the compiler forces ALIGNMENT RULES by stuffing useless
   "padding" bytes into your struct.


THE THREE GOLDEN RULES:
   1. THE K-BYTE RULE: A primitive type of size K must start at a memory offset
      that is a multiple of K.
      - `char` (1 byte): Can go anywhere (0, 1, 2, 3...)
      - `int` (4 bytes): Must start at 0, 4, 8, 12 ...
      - `double` / `pointer` (8 bytes): Must start at 0, 8, 16, 24...
   2. THE ORDER RULE: The compiler will NEVER rearrange your variable to save
      space. It lays them out exactly as you wrote them. 
   3. THE TOTAL SIZE RULE: The total size of the entire struct must be a 
      multiple of the LARGEST alignment requirement inside it ($K_max$). It will
      add padding to the very end of the struct to make this happen. 


WALKING THROUGH EXAMPLE:
   Let's trace... ow the compiler ruins your perfectly good memory with padding:

```c
struct S1 {
    char c;         // 1 byte, needs 1-byte alignment
    int i[2];       // 8 bytes (two 4-byte ints), need 4-byte alignment
    double v;       // 8 bytes, needs 8-byte alignment
}
```

            char c: Starts at offset 0. Takes 1 byte. (Current offset: 1)

            int i[0]: Needs 4-byte alignment. Offset 1 is not a multiple of 4. The compiler adds 3 bytes of padding (offsets 1, 2, 3). i[0] starts at 4. Takes 4 bytes. (Current offset: 8)

            int i[1]: Needs 4-byte alignment. 8 is a multiple of 4. Starts at 8. Takes 4 bytes. (Current offset: 12)

            double v: Needs 8-byte alignment. 12 is not a multiple of 8. The compiler adds 4 bytes of padding (offsets 12, 13, 14, 15). v starts at 16. Takes 8 bytes. (Current offset: 24)

            Final Check: The largest requirement is 8 (double). Is 24 a multiple of 8? Yes. No trailing padding needed.

            Total Size: 24 bytes (with 7 bytes of completely wasted space).


---



### ACCESSING STRUCT FIELDS IN ASSEMBLY
   When you combine arrays with structs -- like an array of student records --
   the compiler has to do a bit of mathematical gymnastics to find the exact
   byte you are asking for.

   ... break down the logic of `a[index].j` using notes...


THE MASTER FORMULA
   Finding a specific field inside an array of structs always follows one strict
   mathematical formula:

        Address = Base Address + (Index * Size of Struct) + Field Offset

   Think of it like deliverying mail in a neighbourhood of identical apartment
   buildings;
      1. BASE ADDRESS: Drive to the street (the start of the array `a`).
      2. Index * Size: Drive past a certain number of buildings to get to the 
         right one (skipping over `index` structs, each of size 12).
      3. Field Offset: Walk up to a specific apartmnet door inside that 
         building (offset `+8` for field `j`)


---
### Breaking Down the `S3` Example
   First, ... the struct from notes to get our numbers:

```c
struct S3 {
    short i;        // 2 bytes. Offset: 0
                    // 2 bytes padding here to align `v`
    int v;          // 4 bytes. Offset: 4
    short j;        // 2 bytes. Offset 8
                    // 2 bytes padding here to make total size a multiple of 4
};
```
   - Size of Struct: 12 bytes
   - Offset of `j`: +8 bytes

   The C code wants to return `a[index].j`. According to our formula, the CPU
   needs to calculate:

   `Address = a + (index * 12) + 8`


---
### The Assembly Translation (The "Multiplier" Trick)
   Here is where the compiler gets incredibly clever. Look at the assembly:

```asm
leaq   (%rdi, %rdi, 2), %rax          # Step 1
movzwl a+8(, %rax, 4), %eax           # Step 2
```
   

Why didn't the compiler just use `12` as the multiplier?
   Because x86-64 hardware literally doesn't allow it! The `Scale` in the 
   `Base(Index, Scale)` addressing mode can ONLY be 1, 2, 4, or 8. It
   physically cannot multiply by 12 in one step.

   So, the compiler factors 12 into 3 * 4 and splits the work into two steps:

   STEP 1: The `leaq` instruction
   `leaq (%rdi, %rdi, 2) %rax`
      - `%rdi` holds our `index`
      - This calculates `%rdi + (%rdi * 2)`, which equals `index * 3`
      - It stores this intermediate result (`index * 3`) in `%rax`.

   STEP 2: The `movzwl` instruction
   `movzwl a+8(, %rax, 4), %eax`
      Let's map this to standard `Imm(Base, Index, Scale)` format:
      - DISPLACEMENT: `a + 8` (The base address of the array, plus the 8-byte
        offset for `j`)
      - BASE: Empty.
      - Index: `%rax` (which currently holds `index * 3`).
      - Scale: `4`.

   Let's put the math together:
   `Address = (a + 8) + (%rax * 4)`
   `Address = a + 8 + ((index * 3) * 4)`
   `Address = a + (index * 12) + 8`

   It perfectly recreated our Master Formula!


---
WHAT IS `movzwl`?
   
   - `mov`: Move data.
   - `z`: Zero-extend (pad the new extra space with zeros, because it's an 
     unsigned or positive number)
   - `w`: Word (The source `j` is a `short`, which is 2 bytes or one "Word")
   - `l`: Long (the destination `%eax` is 4 bytes, meant for standard C `int`).

   The C function returns an `int`, but `j` is only a `short`. So `movzwl` grabs
   the 2-byte `j`, pads it with zeroes to make it 4 bytes, and drops it into the#
   return register.


EXAM TIP FOR STRUCT ARRAYS
   If you see an assembly snippet doing a weird `leaq` multiplication followed 
   by a memory access with a displacement, you are almost certainly looking at 
   an array of structs.
      - The `lea` math + the `Scale` factor will tell you the TOTAL SIZE of the
        struct.
      - The displacement number (like `+8`) will tell you the OFFSET of the 
        specific field being accessed.     


---


byte - word - long - quad           b w l q 
char - short - int - long/double/pointer - long long

---

  Exercise 1: Struct Alignment (from 2023 Q2a - the key exam question)

  Compute the offset of each field, the total size, and the alignment requirement for each
  struct:

  struct d1 { short i; long j; int *k; short *l; };
  struct d2 { char a[3]; char *b[4]; };
  struct d3 { struct d1 c; struct d2 e[2]; };

  Then, given a pointer D3 to struct d3 in %rdi and index i in %rsi:
  - (i) Write one assembly instruction to store D3->c.k into %rax
  - (ii) Write one assembly instruction to store D3->e[i].a (the address) into %rax

  (This is the actual 2023 exam question. Work it out fully before checking the answers PDF.)

---


d1 needs
   - 2 + 6 + 8 + 8 + 8 = 32
   - 3 + 5 + 8*4 = 40
   - 32 + 40 + 40 = 112


```asm
movq 16(%rdi) %rax
```

```asm
movq (%rsi, %rsi, 4), %rax
leaq 32(%rdi, %rax, 8), %rax
```




---
1 2 4 8
byte // word // long // quad
char // short // int // long/pointer/doubel 

---

  Exercise 2: Pointer Arithmetic Table (from 2022 Q2a(i))

  Given int array N at address x_N in %rbx, index i in %rdx. For each assembly line, fill in Type
   (int or pointer), Expression, and Value:

  ┌────────────────────────────┬──────┬────────────┬────────────┐
  │          Assembly          │ Type │ Expression │   Value    │
  ├────────────────────────────┼──────┼────────────┼────────────┤
  │ movl 8(%rbx), %eax         │ int  │ N[2]       │ M[x_N + 8] │
  ├────────────────────────────┼──────┼────────────┼────────────┤
  │ leaq 8(%rbx,%rdx,4), %rax  │ long │ &N[2+i]    │ x_N + 8 + i*4  │
  ├────────────────────────────┼──────┼────────────┼────────────┤
  │ movl 12(%rbx,%rdx,8), %eax │ int  │ N[3+2i]    │ M[x_N + 12 + i*8] │
  ├────────────────────────────┼──────┼────────────┼────────────┤
  │ leaq 20(%rbx,%rdx,4), %rax │ long │ &N[5+i]    │ x_N + 20 + i*4 │
  └────────────────────────────┴──────┴────────────┴────────────┘

  (Hint: leaq computes an address (pointer), movl loads a value (int).)

  Exercise 3: Struct Layout Practice (from lecture slide 19)

  Work out the layout of this struct:

  typedef struct rec {
      int a[4];     // each int = 4 bytes
      long i;       // 8 bytes
      struct rec *next;  // pointer = 8 bytes
  } *rec_p;

  (a) What is the offset of each field? What is the total size?
  (b) Write the assembly for return r->i; given r in %rdi.
  (c) Write the assembly for return r->a[idx]; given r in %rdi, idx in %rsi.


---
(a)
   - int a[4] = 4*4 = 16
   - long i         = 8
   - struct ...     = 8
   total size       = 32

(b)
```asm
movq 16(%rdi), %rax
```

(c)
```asm
movl (%rdi, %rsi, 4), %eax
```

---

  Exercise 4: Nested vs Multi-level Arrays (from lecture slides 12-14)

  Given:
  // Nested (contiguous)
  int pgh[4][5];  // 4 rows, 5 columns of ints

  // Multi-level (pointers)
  int *univ[3] = {mit, cmu, ucb};  // 3 pointers to int arrays

  (a) Write assembly to access pgh[index][digit] given index in %rdi, digit in %rsi. How many
  memory accesses?

  (b) Write assembly to access univ[index][digit] given index in %rdi, digit in %rsi. How many
  memory accesses?

  (c) Why does the nested array need only 1 memory access but the multi-level needs 2?


---



  ---
  Exercise 5: My Own Struct Alignment Practice

  Work these out by hand:

  // (a)
  struct S1 { char c; int i[2]; double v; };

  // (b)
  struct S2 { int x; char c; double d; short s; char *p; };

  // (c) - What if we reorder (b) to minimise padding?
  struct S2_opt { /* reorder the same fields */ };

  For each: draw the byte-by-byte layout, mark padding bytes, and give the total size.